# CE 310 — Week 11 In-Class Exercise
## Hypothesis Testing: Is the Difference Real?

**Dataset:** TxDOT highway construction bid tabulations, 2024–2026
**Points:** 68 (48 auto-graded + 20 manual)

You have two files this week. `CE310_BidItems_2024_2026.csv` holds one row per
line item per bidder — what each contractor offered to charge for a unit of
work. `CE310_BidProjects_2024_2026.csv` holds one row per winning bid, with the
number of firms that competed for it.

Every question below is a question a public agency actually asks: does this
contractor price differently, does this district cost more, does competition

---

**Grading — 100 pts**

| Section | Items | Points |
|---|---|---|
| A — describe and test | 10 numeric | 16 |
| B — compare groups | 12 numeric | 60 |
| Written: B1, B9, B10 | 3 × 4 | 12 |
| Technical memo | 1 | 12 |
| **Total** | | **100** |


## Before You Begin

1. Upload **both** CSV files to this Colab session (folder icon on the left, then upload).
2. Run every cell in order. Cells with `___` need you to fill in the blank first.
3. Do **not** edit or delete any `print(f"ANSWER_... = ...")` line — the grader reads them.
4. Fill in your name in the first cell.

## Setup

In [ ]:
# ── Identify your submission ─────────────────────────────────────────
# Fill both in before you run anything else. NETID is what matches your
# work to your student record — a blank NETID takes a 5-point deduction.
NAME  = ""   # e.g. "Jordan Reyes"
NETID = ""   # e.g. "jreyes"

print(f"NAME  = {NAME}")
print(f"NETID = {NETID}")


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

pd.set_option('display.width', 120)

In [ ]:
items    = pd.read_csv('CE310_BidItems_2024_2026.csv')
projects = pd.read_csv('CE310_BidProjects_2024_2026.csv')

# Most of Section A works with one line item, so the comparison is like-for-like:
# roadway excavation, measured in cubic yards.
exc = items[items['ItemDescription'] == 'EXCAV (ROADWAY)'].copy()

# Ratio of what a contractor bid to what the agency's engineer estimated.
# 1.0 means the bid landed exactly on the estimate.
exc['Ratio'] = exc['UnitPrice_USD'] / exc['EngineerEst_USD']
projects['Ratio'] = projects['WinningBid_USD'] / projects['EngineerEst_USD']

print(f"line items      : {len(items):,} rows")
print(f"roadway excav.  : {len(exc):,} rows")
print(f"winning bids    : {len(projects):,} rows")
exc.head()

## Section A — Hypothesis Testing Foundations

Section A works with roadway excavation unit prices. Every contractor bidding a
project submits a price for the same item, so differences between them are
differences in pricing strategy, not in the work.

### A1 — Within-Group Variability

Before comparing two groups, look at the spread inside one of them. A difference
between group means only matters relative to how much the values scatter.

Compute the standard deviation of unit price among **winning** bidders
(`LowBidder == 'Yes'`).

In [ ]:
low  = exc[exc['LowBidder'] == 'Yes']['UnitPrice_USD']
rest = exc[exc['LowBidder'] == 'No']['UnitPrice_USD']

print(f'winning bids   n = {len(low):,}   mean = ${low.mean():.2f}   median = ${low.median():.2f}')
print(f'losing bids    n = {len(rest):,}   mean = ${rest.mean():.2f}   median = ${rest.median():.2f}')
print()

ANSWER_A1_std_low = round(___, 2)   # fill in: spread of the winning-bid unit prices
print(f'ANSWER_A1_std_low = {ANSWER_A1_std_low}')

### A2 — Two-Sample t-Test: Do Winners Price Lower?

The obvious hypothesis: the contractor who wins the job is the one who priced
this item lower.

**H₀:** mean unit price of winning bids = mean unit price of losing bids
**H₁:** the means differ (two-tailed)
**α = 0.05**

`scipy.stats.ttest_ind(group1, group2)` returns `(t_statistic, p_value)`.

In [ ]:
t_stat, p_val = stats.ttest_ind(___, ___)   # fill in: the two group arrays

print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {p_val:.4f}')
print()

ANSWER_A2_t_stat = round(t_stat, 2)
print(f'ANSWER_A2_t_stat = {ANSWER_A2_t_stat}')

> **Hold on to this result.** Compare the two medians you printed in A1 against
> the p-value you just computed. If they seem to disagree, they do — and B2
> comes back to it. Do not "fix" anything yet.

### A3 — One-Sample t-Test: Do Bids Beat the Engineer's Estimate?

Every TxDOT project carries an engineer's estimate — the agency's own view of
what the work should cost — prepared before bids are opened. The ratio
`UnitPrice ÷ EngineerEstimate` is 1.0 when a bid lands exactly on it.

**H₀:** mean ratio = 1.0 (bids track the estimate on average)
**H₁:** mean ratio ≠ 1.0

`scipy.stats.ttest_1samp(data, popmean)` tests a sample mean against a stated value.

In [ ]:
benchmark = 1.0
t_ratio, p_ratio = stats.ttest_1samp(___, benchmark)   # fill in: the ratio column

print(f'mean ratio   = {exc["Ratio"].mean():.4f}')
print(f'median ratio = {exc["Ratio"].median():.4f}')
print(f't-statistic  = {t_ratio:.4f}')
print(f'p-value      = {p_ratio:.4e}')
print()

ANSWER_A3_t_ratio = round(t_ratio, 2)
print(f'ANSWER_A3_t_ratio = {ANSWER_A3_t_ratio}')

### A4 — One-Way ANOVA: Does Unit Price Differ by District?

TxDOT runs 25 districts. Hauling, local labour markets and competition all vary
between them. A t-test compares two groups; ANOVA compares many at once.

**H₀:** all district mean unit prices are equal
**H₁:** at least one district differs

In [ ]:
groups = [g['UnitPrice_USD'].values
          for _, g in exc.groupby('District') if len(g) >= 30]

F_stat, p_anova = stats.f_oneway(___)   # fill in: unpack the district groups

print(f'districts compared = {len(groups)}')
print(f'F-statistic        = {F_stat:.4f}')
print(f'p-value            = {p_anova:.4e}')
print()

ANSWER_A4_F_district = round(F_stat, 2)
print(f'ANSWER_A4_F_district = {ANSWER_A4_F_district}')

### A5 — Effect Size: Cohen's d

A p-value says whether a difference is distinguishable from zero. It does not
say whether the difference is big. Cohen's d puts the gap between two means in
units of their pooled standard deviation.

| \|d\| | reading |
|---|---|
| < 0.2 | negligible |
| 0.2 – 0.5 | small |
| 0.5 – 0.8 | medium |
| > 0.8 | large |

Compute d for the A2 comparison — winning bids against losing bids.

In [ ]:
n1, s1 = len(low),  low.std()
n2, s2 = len(rest), rest.std()
pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
cohens_d  = (___ - ___) / pooled_sd   # fill in: the two group means

print(f'pooled SD = {pooled_sd:.4f}')
print(f"Cohen's d = {cohens_d:.4f}")
print()

ANSWER_A5_cohens_d = round(cohens_d, 3)
print(f'ANSWER_A5_cohens_d = {ANSWER_A5_cohens_d}')

### A6 — A Share, Not a Mean

A3 asked whether the *average* bid beats the estimate. A procurement officer
asks a blunter question: how often does a bid come in over it?

Compute the fraction of excavation bids with a ratio above 1.0.

In [ ]:
frac_above = (exc['Ratio'] > ___).mean()   # fill in: the break-even ratio

print(f'bids above the engineer estimate = {frac_above:.4f}  ({100*frac_above:.1f}%)')
print()

ANSWER_A6_frac_above = round(frac_above, 4)
print(f'ANSWER_A6_frac_above = {ANSWER_A6_frac_above}')

> Put A3 and A6 side by side. The t-test rejected H₀ decisively, yet fewer than
> half of all bids sit above the estimate and the median ratio is 1.000. Both
> statements are true. B10 asks you to explain how.

### A7 — District Means

Which district pays the most for a cubic yard of excavation, and which the least?

In [ ]:
district_means = exc.groupby(___)[___].mean().round(2).sort_values()   # fill in: grouping key, column

print(district_means)
print()

ANSWER_A7_max_district = district_means.idxmax()
ANSWER_A7_max_mean     = round(district_means.max(), 2)
print(f'ANSWER_A7_max_district = {ANSWER_A7_max_district}')
print(f'ANSWER_A7_max_mean = {ANSWER_A7_max_mean}')

### A8 — Connection to Regression: t-Statistics in OLS

Every regression coefficient carries its own t-statistic testing H₀: the
coefficient is zero. This is the same machinery as A2, applied to a slope.

Bigger jobs usually earn a lower unit price. Fit that on a log–log scale so the
slope reads as an elasticity: the percentage change in price per percentage
change in quantity.

In [ ]:
exc['logq'] = np.log10(exc['Quantity'])
exc['logp'] = np.log10(exc['UnitPrice_USD'])

m = smf.ols(___, data=exc).fit()   # fill in: the model formula, 'outcome ~ predictor'
print(m.summary().tables[1])
print()

ANSWER_A8_slope = round(m.params['logq'], 4)
ANSWER_A8_t     = round(m.tvalues['logq'], 2)
print(f'ANSWER_A8_slope = {ANSWER_A8_slope}')
print(f'ANSWER_A8_t = {ANSWER_A8_t}')

> A slope of about −0.17 means a **ten-fold** larger quantity buys roughly a
> **32% lower** unit price: 10^(−0.17) ≈ 0.68. Economies of scale, measured.

## Section B — Extension Questions

Section A found a price gap between winning bidders and everyone else, and a
district effect behind it. Section B asks whether those results survive contact
with the alternative explanations — quantity, contract type, line item — and then
moves from individual line items to whole awarded contracts, where the agency's
real question lives: does more competition lower what the agency pays? The
section closes on the difference between a result being significant and a result
being worth acting on.

### B1 — Interpretation (Manual · 4 pts)

In 3–4 sentences, state the A2 conclusion the way you would to a procurement
manager. Report the t-statistic, the p-value and Cohen's d from A5, say whether
you reject H₀ at α = 0.05, and say what the result does **not** license you to
conclude. Type your answer in the markdown cell below this one.

*Your answer:*

### B2 — Are Winners Bidding on Different Jobs?

A2 compared prices without asking whether the two groups are bidding on
comparable work. If winning bidders systematically chase larger quantities, and
larger quantities carry lower unit prices (A8), then any price gap could be a
quantity effect wearing a disguise.

Test whether log-quantity differs between winners and the rest.

In [ ]:
q_low  = np.log10(exc[exc['LowBidder'] == 'Yes']['Quantity'])
q_rest = np.log10(exc[exc['LowBidder'] == 'No']['Quantity'])

t_q, p_q = stats.ttest_ind(___, ___, equal_var=False)   # fill in: the two quantity groups
pooled   = np.sqrt(((len(q_low)-1)*q_low.var(ddof=1) + (len(q_rest)-1)*q_rest.var(ddof=1))
                   / (len(q_low) + len(q_rest) - 2))
d_q      = (q_low.mean() - q_rest.mean()) / pooled

print(f't-statistic = {t_q:.4f}   p-value = {p_q:.4f}   Cohen\'s d = {d_q:.4f}')
print()

ANSWER_B2_qty_t = round(t_q, 2)
ANSWER_B2_qty_d = round(d_q, 3)
print(f'ANSWER_B2_qty_t = {ANSWER_B2_qty_t}')
print(f'ANSWER_B2_qty_d = {ANSWER_B2_qty_d}')

### B3 — Construction Against Maintenance

TxDOT lets two kinds of contract. Construction projects build something new;
maintenance contracts keep existing roads serviceable, usually as annual
term agreements. Does excavation cost the same under both?

Work on the log scale — unit prices are right-skewed, and B10 asks you why that
matters.

In [ ]:
constr = np.log10(exc[exc['ProjectType'] == 'Construction']['UnitPrice_USD'])
maint  = np.log10(exc[exc['ProjectType'] == 'Maintenance']['UnitPrice_USD'])

t_type, p_type = stats.ttest_ind(___, ___, equal_var=False)   # fill in: the two project-type groups
pooled = np.sqrt(((len(constr)-1)*constr.var(ddof=1) + (len(maint)-1)*maint.var(ddof=1))
                 / (len(constr) + len(maint) - 2))
d_type = (constr.mean() - maint.mean()) / pooled

print(f'n construction = {len(constr):,}   n maintenance = {len(maint):,}')
print(f't-statistic = {t_type:.4f}   p-value = {p_type:.4e}   Cohen\'s d = {d_type:.4f}')
print()

ANSWER_B3_type_t = round(t_type, 2)
ANSWER_B3_type_d = round(d_type, 3)
print(f'ANSWER_B3_type_t = {ANSWER_B3_type_t}')
print(f'ANSWER_B3_type_d = {ANSWER_B3_type_d}')

### B4 — ANOVA Across Line Items

The extract carries four line items. Do they show the same bid-to-estimate
behaviour, or do contractors price some items more aggressively than others?

**H₀:** the four items have equal mean ratios

In [ ]:
item_groups = [g['UnitPrice_USD'].values / g['EngineerEst_USD'].values
               for _, g in items.groupby('ItemDescription')]

F_item, p_item = stats.f_oneway(___)   # fill in: unpack the item groups

print(items.groupby('ItemDescription').apply(
    lambda g: (g['UnitPrice_USD'] / g['EngineerEst_USD']).mean()).round(4))
print()
print(f'F-statistic = {F_item:.4f}   p-value = {p_item:.4e}')
print()

ANSWER_B4_F_item = round(F_item, 2)
print(f'ANSWER_B4_F_item = {ANSWER_B4_F_item}')

### B5 — Winning Bids by Contract Type

Switch to the project file. Each row is a contract that was awarded, with the
winning bid and the agency's estimate for the whole job.

In [ ]:
ratio_by_type = projects.groupby('ProjectType')['Ratio'].agg(['mean', 'median', 'count']).round(4)
print(ratio_by_type)
print()

# fill in: the two ProjectType labels, as they are spelled in the data
ANSWER_B5_ratio_constr = round(projects[projects['ProjectType'] == ___]['Ratio'].mean(), 4)
ANSWER_B5_ratio_maint  = round(projects[projects['ProjectType'] == ___]['Ratio'].mean(), 4)
print(f'ANSWER_B5_ratio_constr = {ANSWER_B5_ratio_constr}')
print(f'ANSWER_B5_ratio_maint = {ANSWER_B5_ratio_maint}')

# ── B5c: The gap between the two contract types ──────────────────────
# Two ratios are not a finding until you say how far apart they are.
# A negative value means Construction bids come in further below the
# engineer's estimate than Maintenance bids do.

ANSWER_B5_ratio_diff = round(ANSWER_B5_ratio_constr - ANSWER_B5_ratio_maint, 4)

print(f'ANSWER_B5_ratio_diff = {ANSWER_B5_ratio_diff}')


### B6 — ANOVA: Do Districts Win at Different Prices?

A4 tested one line item. This tests whole contracts.

In [ ]:
dist_groups = [g['Ratio'].values
               for _, g in projects.groupby('District') if len(g) >= 30]

F_dist, p_dist = stats.f_oneway(___)   # fill in: unpack the district groups

print(f'districts compared = {len(dist_groups)}')
print(f'F-statistic        = {F_dist:.4f}')
print(f'p-value            = {p_dist:.4e}')
print()

ANSWER_B6_F_district = round(F_dist, 2)
print(f'ANSWER_B6_F_district = {ANSWER_B6_F_district}')

### B7 — Does Competition Save Money?

The question an agency cares about most. Split the awarded contracts into
**thin** competition (3 bidders or fewer) and **deep** competition (6 or more),
and compare what the winning bid cost as a share of the estimate.

**H₀:** the two groups have equal mean ratios

In [ ]:
few  = projects[projects['NumBidders'] <= 3]['Ratio']
many = projects[projects['NumBidders'] >= 6]['Ratio']

t_comp, p_comp = stats.ttest_ind(___, ___, equal_var=False)   # fill in: the two bidder-count groups
pooled = np.sqrt(((len(few)-1)*few.var(ddof=1) + (len(many)-1)*many.var(ddof=1))
                 / (len(few) + len(many) - 2))
d_comp = (few.mean() - many.mean()) / pooled

print(f'<= 3 bidders  n = {len(few):,}   mean ratio = {few.mean():.4f}')
print(f'>= 6 bidders  n = {len(many):,}   mean ratio = {many.mean():.4f}')
print(f't-statistic = {t_comp:.4f}   p-value = {p_comp:.4e}   Cohen\'s d = {d_comp:.4f}')
print()

ANSWER_B7_t = round(t_comp, 2)
ANSWER_B7_d = round(d_comp, 3)
print(f'ANSWER_B7_t = {ANSWER_B7_t}')
print(f'ANSWER_B7_d = {ANSWER_B7_d}')

### B8 — How Often Does the Winner Beat the Estimate?

A6 asked this of individual line items. Ask it of awarded contracts.

In [ ]:
frac_over = (projects['Ratio'] > ___).mean()   # fill in: the break-even ratio

print(f'awarded contracts above the engineer estimate = {frac_over:.4f}  ({100*frac_over:.1f}%)')
print()

ANSWER_B8_frac_over = round(frac_over, 4)
print(f'ANSWER_B8_frac_over = {ANSWER_B8_frac_over}')

### B9 — Post-Hoc Analysis: Tukey HSD (Manual · 4 pts)

A4 told you *some* district differs. It did not say which. Tukey's HSD tests
every pair while holding the family-wise error rate at α.

Run the cell, then answer in the markdown cell below: how many pairs are
significant out of how many tested, which pair shows the largest mean
difference, and why testing 300 pairs at α = 0.05 without a correction would be
a mistake.

In [ ]:
tukey = pairwise_tukeyhsd(exc['UnitPrice_USD'], exc['District'], alpha=0.05)
res   = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])

print(f'pairs tested     = {len(res)}')
print(f'pairs significant = {(res["reject"] == True).sum()}')
print()
print(res.assign(absdiff=res['meandiff'].astype(float).abs())
         .sort_values('absdiff', ascending=False)
         .head(8)[['group1', 'group2', 'meandiff', 'p-adj', 'reject']])

*Your answer:*

### B10 — Written Reflection: Statistical vs. Practical Significance (Manual · 4 pts)

Two results in this in-class exercise pull in opposite directions.

- **A3** rejected H₀ that the mean bid-to-estimate ratio is 1.0, with a p-value
  far below any threshold you would ever set.
- **A6** found that fewer than half of all bids exceed the estimate, and the
  median ratio is exactly 1.000.

In 100–150 words: explain how both can be true at once, name the feature of the
distribution that produces it, and say which of the two numbers you would put in
a briefing to a procurement director — and why.

*Your answer:*

## Memo (Manual · 12 pts)

Write a short technical memo (To / From / Date / Subject, then three paragraphs)
to a district engineer who has asked one question: **what would actually reduce
what we pay?**

- **Paragraph 1** — what the bidding data shows about price variation. Cite A4
  (district F-statistic) and A7 (the highest and lowest district means).
- **Paragraph 2** — the competition result from B7. Cite the two group means,
  the t-statistic and Cohen's d, and translate the ratio gap into what it means
  for a contract of typical size.
- **Paragraph 3** — one recommendation, and one honest limitation of what
  observational bid data can support. These are not randomised trials; districts
  differ in more ways than bidder counts.

## Looking Ahead: Week 12 — Sensitivity Analysis

A8 fitted a line through bid prices. Week 12 builds a cost forecast on that line
and asks which of its four inputs — quantity, unit price, overhead, escalation —
actually decides the answer. It is not the one the forecast responds to most
steeply. It is the one you know least well.

## Before You Submit

- [ ] `NAME` and `NETID` filled in at the top and showing in the cell output — a blank `NETID` takes a 5-point deduction
- [ ] Every cell run in order, top to bottom (Runtime → Run all) — no errors, no cell left unrun
- [ ] Every `ANSWER_` line prints a value, and no `print(f"ANSWER_... = ...")` line edited or deleted
- [ ] Every `___` blank is replaced
- [ ] B1, B9, B10 and the memo are written in their markdown cells
- [ ] Downloaded as `.ipynb` — not PDF, not `.py`
- [ ] Renamed `CE310_W11_<NetID>.ipynb` and uploaded to the Week 11 D2L dropbox by **9:00 AM next Wednesday**